In [1]:
from pyspark.sql import functions as F

# 1. Odczyt surowych plików Parquet z Lakehouse_Bronze
bronze_taxi_path = "abfss://2f59ccc2-e9d6-44b5-8c0c-1bdbf1fe148a@onelake.dfs.fabric.microsoft.com/33b93b7e-0899-43c6-81a8-654bc740aa32/Files/nyc_taxi"

df_raw = spark.read.option("mergeSchema", "true").parquet(bronze_taxi_path)

# 2. Standaryzacja nazw kolumn do małych liter
df_standardized = df_raw.toDF(*[c.lower() for c in df_raw.columns])

# 3. Czyszczenie, walidacja i wzbogacenie danych (warstwa Silver)
df_silver = (
    df_standardized
    .withColumn("tpep_pickup_datetime", F.to_timestamp("tpep_pickup_datetime"))
    .withColumn("tpep_dropoff_datetime", F.to_timestamp("tpep_dropoff_datetime"))
    .withColumn(
        "trip_duration_minutes",
        F.round(
            (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60.0,
            2
        )
    )
    # Odfiltrowanie nieprawidłowych rekordów i wartości skrajnych
    .filter(
        (F.col("tpep_pickup_datetime").isNotNull()) &
        (F.col("tpep_dropoff_datetime").isNotNull()) &
        (F.col("trip_duration_minutes") > 0) &
        (F.col("trip_duration_minutes") < 1440) &
        (F.col("trip_distance") > 0) &
        (F.col("fare_amount") > 0) &
        (F.col("total_amount") > 0) &
        (F.col("passenger_count") > 0) &
        (F.year("tpep_pickup_datetime").between(2020, 2026))
    )
    # Dodanie kolumn czasowych pod przyszłe łączenie z danymi FX i GDP
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .withColumn("pickup_year", F.year("tpep_pickup_datetime"))
    .withColumn("pickup_month", F.month("tpep_pickup_datetime"))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .dropDuplicates()
)

# 4. Zapis oczyszczonej tabeli Delta do domyślnego Lakehouse_Silver
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("nyc_taxi_silver")
)

print(f"Zapisano tabelę nyc_taxi_silver. Liczba poprawnych rekordów: {df_silver.count()}")
display(df_silver.limit(5))

StatementMeta(, 6dd049b1-1c8c-47d6-8d8d-6326876ee6e7, 3, Finished, Available, Finished, False)

Zapisano tabelę nyc_taxi_silver. Liczba poprawnych rekordów: 2723735


SynapseWidget(Synapse.DataFrame, 16452920-32ec-4397-9bae-49134a209e69)